# Forward pass e backpropagation

Este notebook implementa em python e usando NumPy, uma rede neural para **classificar as três espécies do conjunto Iris**. 
A rede tem duas camadas escondidas e arquitetura `4 -> 4 -> 4 -> 3`.


## 1. Dados

Cada flor possui quatro medidas e pertence a uma de três classes. O conjunto é dividido de forma estratificada em 80% para treinamento e 20% para teste, preservando a proporção das classes. A padronização usa apenas a média e o desvio-padrão do treinamento.

In [25]:
import csv

import numpy as np

np.set_printoptions(precision=5, suppress=True)

with open('iris_dataset.csv', encoding='utf-8', newline='') as arquivo:
    flores = list(csv.DictReader(arquivo))

todas_as_entradas = np.array([
    [
        float(flor['sepal.length']),
        float(flor['sepal.width']),
        float(flor['petal.length']),
        float(flor['petal.width']),
    ]
    for flor in flores
])

nomes_das_classes = sorted({flor['variety'] for flor in flores})
classe_para_indice = {nome: indice for indice, nome in enumerate(nomes_das_classes)}
todas_as_classes = np.array([
    classe_para_indice[flor['variety']] for flor in flores
])
todas_as_saidas_esperadas = np.eye(3)[todas_as_classes]

gerador_da_divisao = np.random.default_rng(42)
indices_de_treino = []
indices_de_teste = []

for i in range(3):
    i = np.flatnonzero(todas_as_classes == i)
    gerador_da_divisao.shuffle(i)
    quantidade_de_treino = int(0.8 * len(i))
    indices_de_treino.extend(i[:quantidade_de_treino])
    indices_de_teste.extend(i[quantidade_de_treino:])

indices_de_treino = np.array(indices_de_treino)
indices_de_teste = np.array(indices_de_teste)
gerador_da_divisao.shuffle(indices_de_treino)
gerador_da_divisao.shuffle(indices_de_teste)

entradas_de_treino = todas_as_entradas[indices_de_treino]
saidas_esperadas_de_treino = todas_as_saidas_esperadas[indices_de_treino]
classes_de_treino = todas_as_classes[indices_de_treino]
entradas_de_teste = todas_as_entradas[indices_de_teste]
saidas_esperadas_de_teste = todas_as_saidas_esperadas[indices_de_teste]
classes_de_teste = todas_as_classes[indices_de_teste]

# As estatísticas do teste não participam do treinamento.
media_dos_atributos = entradas_de_treino.mean(axis=0)
desvio_padrao_dos_atributos = entradas_de_treino.std(axis=0)
entradas_de_treino = (
    entradas_de_treino - media_dos_atributos
) / desvio_padrao_dos_atributos
entradas_de_teste = (
    entradas_de_teste - media_dos_atributos
) / desvio_padrao_dos_atributos

print('Entradas de treinamento:', entradas_de_treino.shape)
print('Entradas de teste:', entradas_de_teste.shape)
print('Exemplos por classe no treinamento:', np.bincount(classes_de_treino))
print('Exemplos por classe no teste:', np.bincount(classes_de_teste))
print('Classes:', nomes_das_classes)
print('Primeira entrada de treinamento padronizada:', entradas_de_treino[0])
print('Primeira saída esperada:', saidas_esperadas_de_treino[0])

assert entradas_de_treino.shape == (120, 4)
assert entradas_de_teste.shape == (30, 4)
assert saidas_esperadas_de_treino.shape == (120, 3)
assert saidas_esperadas_de_teste.shape == (30, 3)
assert np.all(np.bincount(classes_de_treino) == 40)
assert np.all(np.bincount(classes_de_teste) == 10)
assert not np.intersect1d(indices_de_treino, indices_de_teste).size

Entradas de treinamento: (120, 4)
Entradas de teste: (30, 4)
Exemplos por classe no treinamento: [40 40 40]
Exemplos por classe no teste: [10 10 10]
Classes: ['Setosa', 'Versicolor', 'Virginica']
Primeira entrada de treinamento padronizada: [0.52293 0.53151 1.25764 1.70982]
Primeira saída esperada: [0. 0. 1.]


## 2. Arquitetura e funções de ativação

A rede possui três camadas de pesos:

1. quatro entradas para quatro neurônios escondidos;
2. quatro ativações para quatro neurônios escondidos;
3. quatro ativações para três classes de saída.

As duas camadas escondidas usam a sigmoide. A saída usa softmax, que produz probabilidades positivas cuja soma é um.

In [58]:
def sigmoide(valor):
    valor = np.clip(valor, -500.0, 500.0)
    return 1.0 / (1.0 + np.exp(-valor))


def derivada_da_sigmoide(ativacao):
    return ativacao * (1.0 - ativacao)


def softmax(valores):
    valores = valores - np.max(valores)
    exponenciais = np.exp(valores)
    return exponenciais / np.sum(exponenciais)


def inicializar_parametros(semente=42):
    gerador = np.random.default_rng(semente)
    tamanhos = [4, 4, 4, 3]
    pesos = []
    vieses = []

    for tamanho_anterior, tamanho_atual in zip(tamanhos[:-1], tamanhos[1:]):
        pesos.append(
            gerador.normal(
                0.0,
                1.0 / np.sqrt(tamanho_anterior),
                size=(tamanho_atual, tamanho_anterior),
            )
        )
        vieses.append(np.zeros((tamanho_atual, 1)))

    return pesos, vieses


pesos_iniciais, vieses_iniciais = inicializar_parametros()

for i, (peso, vies) in enumerate(zip(pesos_iniciais, vieses_iniciais)):
    print(f'\n### Camada {i+1}:')
    print(f'\nDimensões dos pesos: {peso.shape}')
    print(f'Valores dos pesos:\n{peso}')
    print(f'\nDimensões dos vieses: {vies.shape}')
    print(f'Valores dos vieses:\n{vies}')


### Camada 1:

Dimensões dos pesos: (4, 4)
Valores dos pesos:
[[ 0.15236 -0.51999  0.37523  0.47028]
 [-0.97552 -0.65109  0.06392 -0.15812]
 [-0.0084  -0.42652  0.4397   0.3889 ]
 [ 0.03302  0.56362  0.23375 -0.42965]]

Dimensões dos vieses: (4, 1)
Valores dos vieses:
[[0.]
 [0.]
 [0.]
 [0.]]

### Camada 2:

Dimensões dos pesos: (4, 4)
Valores dos pesos:
[[ 0.18438 -0.47944  0.43923 -0.02496]
 [-0.09243 -0.34046  0.61127 -0.07726]
 [-0.21416 -0.17607  0.26615  0.18272]
 [ 0.20637  0.21541  1.07082 -0.20321]]

Dimensões dos vieses: (4, 1)
Valores dos vieses:
[[0.]
 [0.]
 [0.]
 [0.]]

### Camada 3:

Dimensões dos pesos: (3, 4)
Valores dos pesos:
[[-0.25612 -0.40689  0.30799  0.56449]
 [-0.05697 -0.42008 -0.41224  0.3253 ]
 [ 0.37163  0.27158 -0.33275  0.11608]]

Dimensões dos vieses: (3, 1)
Valores dos vieses:
[[0.]
 [0.]
 [0.]]


## 3. Forward pass

Em cada camada escondida calculamos `z = W @ a_anterior + b` e depois `a = sigmoide(z)`. Na camada de saída, a ativação é `softmax(z)`.

As contas das três camadas estão escritas separadamente para que o caminho da entrada até a saída permaneça evidente.

In [65]:
def passagem_para_frente(entrada, pesos, vieses):
    ativacao_0 = np.asarray(entrada, dtype=float).reshape(4, 1)

    combinacao_1 = pesos[0] @ ativacao_0 + vieses[0]
    ativacao_1 = sigmoide(combinacao_1)

    combinacao_2 = pesos[1] @ ativacao_1 + vieses[1]
    ativacao_2 = sigmoide(combinacao_2)

    combinacao_3 = pesos[2] @ ativacao_2 + vieses[2]
    probabilidades = softmax(combinacao_3)

    ativacoes = [ativacao_0, ativacao_1, ativacao_2, probabilidades]
    combinacoes = [combinacao_1, combinacao_2, combinacao_3]
    return ativacoes, combinacoes


ativacoes_do_exemplo, combinacoes_do_exemplo = passagem_para_frente(
    entradas_de_treino[0], pesos_iniciais, vieses_iniciais
)

In [71]:
ativacoes_do_exemplo

[array([[0.52293],
        [0.53151],
        [1.25764],
        [1.70982]]),
 array([[0.74636],
        [0.25996],
        [0.72846],
        [0.46909]]),
 array([[0.57962],
        [0.56256],
        [0.51849],
        [0.70987]]),
 array([[0.36406],
        [0.23559],
        [0.40034]])]

In [73]:
combinacoes_do_exemplo

[array([[ 1.07929],
        [-1.04616],
        [ 0.98683],
        [-0.1238 ]]),
 array([[0.32122],
        [0.25155],
        [0.07398],
        [0.89475]]),
 array([[ 0.18305],
        [-0.25216],
        [ 0.27805]])]

### Forward pass de uma flor, passo a passo

A célula seguinte mostra os operandos e o resultado de cada conta. Nas duas primeiras camadas, `a` é a sigmoide de `z`; na última, `a` contém as probabilidades produzidas pela softmax.

In [4]:
nomes_das_camadas = ['primeira escondida', 'segunda escondida', 'saída']

for indice, nome in enumerate(nomes_das_camadas):
    print(f'\nCAMADA {indice + 1}: {nome}')
    print(f'W{indice + 1} =\n{pesos_iniciais[indice]}')
    print(f'a{indice} =\n{ativacoes_do_exemplo[indice]}')
    produto = pesos_iniciais[indice] @ ativacoes_do_exemplo[indice]
    print(f'W{indice + 1} @ a{indice} =\n{produto}')
    print(f'b{indice + 1} =\n{vieses_iniciais[indice]}')
    print(f'z{indice + 1} = W{indice + 1} @ a{indice} + b{indice + 1} =\n{combinacoes_do_exemplo[indice]}')
    print(f'a{indice + 1} =\n{ativacoes_do_exemplo[indice + 1]}')

print('\nSoma das probabilidades:', ativacoes_do_exemplo[-1].sum())


CAMADA 1: primeira escondida
W1 =
[[ 0.15236 -0.51999  0.37523  0.47028]
 [-0.97552 -0.65109  0.06392 -0.15812]
 [-0.0084  -0.42652  0.4397   0.3889 ]
 [ 0.03302  0.56362  0.23375 -0.42965]]
a0 =
[[0.52293]
 [0.53151]
 [1.25764]
 [1.70982]]
W1 @ a0 =
[[ 1.07929]
 [-1.04616]
 [ 0.98683]
 [-0.1238 ]]
b1 =
[[0.]
 [0.]
 [0.]
 [0.]]
z1 = W1 @ a0 + b1 =
[[ 1.07929]
 [-1.04616]
 [ 0.98683]
 [-0.1238 ]]
a1 =
[[0.74636]
 [0.25996]
 [0.72846]
 [0.46909]]

CAMADA 2: segunda escondida
W2 =
[[ 0.18438 -0.47944  0.43923 -0.02496]
 [-0.09243 -0.34046  0.61127 -0.07726]
 [-0.21416 -0.17607  0.26615  0.18272]
 [ 0.20637  0.21541  1.07082 -0.20321]]
a1 =
[[0.74636]
 [0.25996]
 [0.72846]
 [0.46909]]
W2 @ a1 =
[[0.32122]
 [0.25155]
 [0.07398]
 [0.89475]]
b2 =
[[0.]
 [0.]
 [0.]
 [0.]]
z2 = W2 @ a1 + b2 =
[[0.32122]
 [0.25155]
 [0.07398]
 [0.89475]]
a2 =
[[0.57962]
 [0.56256]
 [0.51849]
 [0.70987]]

CAMADA 3: saída
W3 =
[[-0.25612 -0.40689  0.30799  0.56449]
 [-0.05697 -0.42008 -0.41224  0.3253 ]
 [ 0.3716

## 4. Verossimilhança e loss

Se a classe correta é `k`, a verossimilhança do exemplo é a probabilidade `p[k]` atribuída pela rede a essa classe. Para todo o conjunto, a verossimilhança é o produto dessas probabilidades:

```text
V = produto(p_classe_correta)
```

Produtos de muitos números pequenos são difíceis de representar numericamente. Por isso, usamos o logaritmo e minimizamos a log-verossimilhança negativa média. Com uma saída esperada one-hot, essa função também é chamada de entropia cruzada categórica:

```text
perda = -soma(saida_esperada * log(probabilidades))
```

Minimizar essa perda é equivalente a maximizar a verossimilhança.

In [80]:
def calcular_perda(probabilidades, saida_esperada):
    saida_esperada = np.asarray(saida_esperada, dtype=float).reshape(3, 1)
    probabilidades = np.clip(probabilidades, 1e-12, 1.0)
    return -float(np.sum(saida_esperada * np.log(probabilidades)))


probabilidades_do_exemplo = ativacoes_do_exemplo[-1]
saida_esperada_do_exemplo = saidas_esperadas_de_treino[0].reshape(3, 1)
indice_correto = classes_de_treino[0]
verossimilhanca_do_exemplo = probabilidades_do_exemplo[indice_correto, 0]
perda_do_exemplo = calcular_perda(
    probabilidades_do_exemplo, saida_esperada_do_exemplo
)

print('Saída esperada:', saida_esperada_do_exemplo.ravel())
print('Probabilidades:', probabilidades_do_exemplo.ravel())
print('Verossimilhança do exemplo:', verossimilhanca_do_exemplo)
print('Log-verossimilhança:', np.log(verossimilhanca_do_exemplo))
print('Perda = -log-verossimilhança:', perda_do_exemplo)

Saída esperada: [0. 0. 1.]
Probabilidades: [0.36406 0.23559 0.40034]
Verossimilhança do exemplo: 0.40034498909741756
Log-verossimilhança: -0.9154286308465133
Perda = -log-verossimilhança: 0.9154286308465133


## 5. Backpropagation



In [83]:
def retropropagacao(saida_esperada, pesos, ativacoes):
    saida_esperada = np.asarray(saida_esperada, dtype=float).reshape(3, 1)
    ativacao_0, ativacao_1, ativacao_2, probabilidades = ativacoes

    delta_3 = probabilidades - saida_esperada
    gradiente_pesos_3 = delta_3 @ ativacao_2.T
    gradiente_vies_3 = delta_3

    gradiente_ativacao_2 = pesos[2].T @ delta_3
    delta_2 = gradiente_ativacao_2 * derivada_da_sigmoide(ativacao_2)
    gradiente_pesos_2 = delta_2 @ ativacao_1.T
    gradiente_vies_2 = delta_2

    gradiente_ativacao_1 = pesos[1].T @ delta_2
    delta_1 = gradiente_ativacao_1 * derivada_da_sigmoide(ativacao_1)
    gradiente_pesos_1 = delta_1 @ ativacao_0.T
    gradiente_vies_1 = delta_1

    gradientes_dos_pesos = [
        gradiente_pesos_1, gradiente_pesos_2, gradiente_pesos_3
    ]
    gradientes_dos_vieses = [
        gradiente_vies_1, gradiente_vies_2, gradiente_vies_3
    ]
    deltas = [delta_1, delta_2, delta_3]
    return gradientes_dos_pesos, gradientes_dos_vieses, deltas


gradientes_pesos_exemplo, gradientes_vieses_exemplo, deltas_exemplo = (
    retropropagacao(
        saida_esperada_do_exemplo,
        pesos_iniciais,
        ativacoes_do_exemplo,
    )
)

### Backpropagation do mesmo exemplo

Da camada de saída é propagada o gradiente até a primeira camada escondida.

In [86]:
probabilidades = ativacoes_do_exemplo[-1]
gradiente_da_perda_nas_probabilidades = (
    -saida_esperada_do_exemplo / probabilidades
)
jacobiana_da_softmax = (
    np.diagflat(probabilidades) - probabilidades @ probabilidades.T
)
delta_3_passo_a_passo = (
    jacobiana_da_softmax.T @ gradiente_da_perda_nas_probabilidades
)

print('CAMADA 3: saída')
print('dL/dp = -y / p =')
print(gradiente_da_perda_nas_probabilidades)
print('Jacobiana da softmax = diag(p) - p @ p.T =')
print(jacobiana_da_softmax)
print('Jacobiana.T @ dL/dp =')
print(delta_3_passo_a_passo)
print('delta3 = probabilidades - saída esperada =')
print(deltas_exemplo[2])
print('gradiente recebido pela camada 2 = W3.T @ delta3 =')
print(pesos_iniciais[2].T @ deltas_exemplo[2])

print('\nCAMADA 2: segunda escondida')
print('delta2 = gradiente recebido * derivada da sigmoide =')
print(deltas_exemplo[1])
print('gradiente recebido pela camada 1 = W2.T @ delta2 =')
print(pesos_iniciais[1].T @ deltas_exemplo[1])

print('\nCAMADA 1: primeira escondida')
print('delta1 = gradiente recebido * derivada da sigmoide =')
print(deltas_exemplo[0])

for indice in reversed(range(3)):
    print(f'\nGradientes da camada {indice + 1}')
    print(f'dL/dW{indice + 1} = delta{indice + 1} @ a{indice}.T =')
    print(gradientes_pesos_exemplo[indice])
    print(f'dL/db{indice + 1} = delta{indice + 1} =')
    print(gradientes_vieses_exemplo[indice])

assert np.allclose(delta_3_passo_a_passo, deltas_exemplo[2])

CAMADA 3: saída
dL/dp = -y / p =
[[-0.     ]
 [-0.     ]
 [-2.49785]]
Jacobiana da softmax = diag(p) - p @ p.T =
[[ 0.23152 -0.08577 -0.14575]
 [-0.08577  0.18009 -0.09432]
 [-0.14575 -0.09432  0.24007]]
Jacobiana.T @ dL/dp =
[[ 0.36406]
 [ 0.23559]
 [-0.59966]]
delta3 = probabilidades - saída esperada =
[[ 0.36406]
 [ 0.23559]
 [-0.59966]]
gradiente recebido pela camada 2 = W3.T @ delta3 =
[[-0.32951]
 [-0.40995]
 [ 0.21454]
 [ 0.21254]]

CAMADA 2: segunda escondida
delta2 = gradiente recebido * derivada da sigmoide =
[[-0.08029]
 [-0.10088]
 [ 0.05356]
 [ 0.04377]]
gradiente recebido pela camada 1 = W2.T @ delta2 =
[[-0.00792]
 [ 0.07284]
 [-0.0358 ]
 [ 0.01069]]

CAMADA 1: primeira escondida
delta1 = gradiente recebido * derivada da sigmoide =
[[-0.0015 ]
 [ 0.01401]
 [-0.00708]
 [ 0.00266]]

Gradientes da camada 3
dL/dW3 = delta3 @ a2.T =
[[ 0.21102  0.20481  0.18876  0.25844]
 [ 0.13656  0.13253  0.12215  0.16724]
 [-0.34757 -0.33734 -0.31091 -0.42568]]
dL/db3 = delta3 =
[[ 0.3640

## 6. Conferência dos gradientes



In [89]:
def perda_de_um_exemplo(entrada, saida_esperada, pesos, vieses):
    ativacoes, _ = passagem_para_frente(entrada, pesos, vieses)
    return calcular_perda(ativacoes[-1], saida_esperada)


def calcular_gradientes_numericos(entrada, saida_esperada, pesos, vieses):
    passo = 1e-5
    numericos_pesos = [np.zeros_like(peso) for peso in pesos]
    numericos_vieses = [np.zeros_like(vies) for vies in vieses]

    for parametros, gradientes in [
        (pesos, numericos_pesos),
        (vieses, numericos_vieses),
    ]:
        for indice_da_camada, parametro in enumerate(parametros):
            for posicao in np.ndindex(parametro.shape):
                valor_original = parametro[posicao]

                parametro[posicao] = valor_original + passo
                perda_a_frente = perda_de_um_exemplo(
                    entrada, saida_esperada, pesos, vieses
                )

                parametro[posicao] = valor_original - passo
                perda_atras = perda_de_um_exemplo(
                    entrada, saida_esperada, pesos, vieses
                )

                parametro[posicao] = valor_original
                gradientes[indice_da_camada][posicao] = (
                    perda_a_frente - perda_atras
                ) / (2.0 * passo)

    return numericos_pesos, numericos_vieses


numericos_pesos, numericos_vieses = calcular_gradientes_numericos(
    entradas_de_treino[0],
    saidas_esperadas_de_treino[0],
    pesos_iniciais,
    vieses_iniciais,
)
maior_diferenca = max(
    np.max(np.abs(analitico - numerico))
    for analitico, numerico in zip(
        gradientes_pesos_exemplo + gradientes_vieses_exemplo,
        numericos_pesos + numericos_vieses,
    )
)

print(f'Maior diferença entre os gradientes: {maior_diferenca:.2e}')
assert maior_diferenca < 1e-6

Maior diferença entre os gradientes: 1.65e-11


## 7. Treinamento

Forward e o backward das 120 flores do conjunto de treinamento (80%)
Calculo da média dos gradientes e atualizamos os parâmetros na direção que reduz a perda

In [92]:
def calcular_gradientes_medios(entradas, saidas_esperadas, pesos, vieses):
    soma_pesos = [np.zeros_like(peso) for peso in pesos]
    soma_vieses = [np.zeros_like(vies) for vies in vieses]
    soma_das_perdas = 0.0

    for entrada, saida_esperada in zip(entradas, saidas_esperadas):
        ativacoes, _ = passagem_para_frente(entrada, pesos, vieses)
        soma_das_perdas += calcular_perda(ativacoes[-1], saida_esperada)
        gradientes_pesos, gradientes_vieses, _ = retropropagacao(
            saida_esperada, pesos, ativacoes
        )

        for indice in range(3):
            soma_pesos[indice] += gradientes_pesos[indice]
            soma_vieses[indice] += gradientes_vieses[indice]

    quantidade = len(entradas)
    perda_media = soma_das_perdas / quantidade
    media_pesos = [gradiente / quantidade for gradiente in soma_pesos]
    media_vieses = [gradiente / quantidade for gradiente in soma_vieses]
    return perda_media, media_pesos, media_vieses


def atualizar_parametros(pesos, vieses, gradientes_pesos, gradientes_vieses, taxa):
    pesos = [
        peso - taxa * gradiente
        for peso, gradiente in zip(pesos, gradientes_pesos)
    ]
    vieses = [
        vies - taxa * gradiente
        for vies, gradiente in zip(vieses, gradientes_vieses)
    ]
    return pesos, vieses

In [94]:
pesos, vieses = inicializar_parametros()
taxa_de_aprendizagem = 1.0
quantidade_de_epocas = 1500
historico = []

for epoca in range(quantidade_de_epocas):
    perda_media, gradientes_pesos, gradientes_vieses = calcular_gradientes_medios(
        entradas_de_treino, saidas_esperadas_de_treino, pesos, vieses
    )

    if epoca % 300 == 0:
        historico.append((epoca, perda_media))

    pesos, vieses = atualizar_parametros(
        pesos,
        vieses,
        gradientes_pesos,
        gradientes_vieses,
        taxa_de_aprendizagem,
    )

perda_final, _, _ = calcular_gradientes_medios(
    entradas_de_treino, saidas_esperadas_de_treino, pesos, vieses
)
historico.append((quantidade_de_epocas, perda_final))

for epoca, perda in historico:
    print(f'Época {epoca:4d}: perda média = {perda:.6f}')

assert historico[-1][1] < historico[0][1]

Época    0: perda média = 1.110182
Época  300: perda média = 0.083554
Época  600: perda média = 0.053485
Época  900: perda média = 0.047866
Época 1200: perda média = 0.045109
Época 1500: perda média = 0.043131


## 7. Resultado

A classe escolhida é a posição da maior probabilidade (argmax). A acurácia de treinamento mostra o ajuste aos dados usados para calcular os gradientes; a acurácia de teste mede o desempenho nos 20% de exemplos que permaneceram fora do treinamento.

In [97]:
def avaliar(entradas, classes_corretas, pesos, vieses):
    probabilidades = np.vstack([
        passagem_para_frente(entrada, pesos, vieses)[0][-1].ravel()
        for entrada in entradas
    ])
    classes_preditas = np.argmax(probabilidades, axis=1)
    acuracia = np.mean(classes_preditas == classes_corretas)

    matriz_de_confusao = np.zeros((3, 3), dtype=int)
    for classe_correta, classe_predita in zip(
        classes_corretas, classes_preditas
    ):
        matriz_de_confusao[classe_correta, classe_predita] += 1

    return acuracia, matriz_de_confusao


acuracia_de_treino, matriz_de_confusao_de_treino = avaliar(
    entradas_de_treino, classes_de_treino, pesos, vieses
)
acuracia_de_teste, matriz_de_confusao_de_teste = avaliar(
    entradas_de_teste, classes_de_teste, pesos, vieses
)

print('Ordem das classes:', nomes_das_classes)
print(f'Acurácia de treinamento: {acuracia_de_treino:.2%}')
print('Matriz de confusão do treinamento:')
print(matriz_de_confusao_de_treino)
print(f'\nAcurácia de teste: {acuracia_de_teste:.2%}')
print('Matriz de confusão do teste:')
print(matriz_de_confusao_de_teste)

assert acuracia_de_treino >= 0.85
assert acuracia_de_teste >= 0.80

Ordem das classes: ['Setosa', 'Versicolor', 'Virginica']
Acurácia de treinamento: 98.33%
Matriz de confusão do treinamento:
[[40  0  0]
 [ 0 39  1]
 [ 0  1 39]]

Acurácia de teste: 96.67%
Matriz de confusão do teste:
[[10  0  0]
 [ 0 10  0]
 [ 0  1  9]]
